# AI-Based Adaptive Mock Interview System — Updated

**Dataset**: `interview_dataset_final_clean_1663rows.csv` (1663 rows, 9 skills, 3 difficulty levels)

**Skills covered**: Python, OOP, DBMS, DSA, ML, CN, OS, System Design, HR

**Updates from original notebook**:
- Uses real 1663-row dataset instead of synthetic generated data
- Question bank built directly from dataset (583 unique questions)
- Skill selection added — user picks a topic before interview starts
- `ColumnTransformer` combines TF-IDF text features + 4 numeric features (`cos_similarity`, `length_ratio`, `aligned_score`, `word_count`)
- `ComplementNB` replaces `MultinomialNB` (fixes negative value crash with TF-IDF)
- `MinMaxScaler` for NB, `StandardScaler` for LR/SVM/RF
- `StratifiedKFold` with `class_weight='balanced'` throughout
- Best model auto-selected from cross-validation (not hardcoded)
- `ideal_answer` from dataset used as golden reference for NLP scoring
- Session summary with per-round breakdown at end

In [ ]:
# =========================
# MODULE 1: Install & Import
# =========================
!pip install -q scikit-learn pandas matplotlib sentence-transformers

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import re
import joblib

from sklearn.model_selection import cross_val_score, cross_validate, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB

print("All imports successful.")

In [ ]:
# =========================
# MODULE 2: Load Dataset
# =========================
# Upload your file if running in Colab
# from google.colab import files
# files.upload()

df = pd.read_csv("interview_dataset_final_clean_1663rows.csv")

print("Columns      :", df.columns.tolist())
print("Shape        :", df.shape)
print("\nLabel Distribution:\n", df['label'].value_counts())
print("\nSkills       :", df['skill'].unique().tolist())
print("Difficulties :", df['difficulty'].unique().tolist())
print("\nUnique questions:", df['question'].nunique())
print("\nPer skill-difficulty question count:")
print(df.groupby(['skill','difficulty'])['question'].nunique().to_string())

In [ ]:
# =========================
# MODULE 3: NLP Scoring Engine
# =========================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
    USE_EMBEDDINGS = True
    print("Semantic embeddings: ENABLED")
except ImportError:
    USE_EMBEDDINGS = False
    print("Semantic embeddings: DISABLED (install sentence-transformers to enable)")


class NLPScorer:
    """
    Scores a user's answer against the ideal answer using:
      - Keyword match
      - Length heuristic
      - TF-IDF cosine similarity
      - Semantic similarity (sentence-transformers, if available)
    """
    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words='english')

    def score_keyword_match(self, answer, expected_keywords):
        if not expected_keywords:
            return 1.0
        text = answer.lower()
        matched = sum(1 for kw in expected_keywords
                      if re.search(r'\b' + re.escape(kw.lower()) + r'\b', text))
        return matched / len(expected_keywords)

    def score_length(self, answer, ideal_length=20):
        return min(len(answer.split()) / ideal_length, 1.0)

    def score_tfidf_similarity(self, answer, golden_answer):
        try:
            tfidf = self.vectorizer.fit_transform([golden_answer, answer])
            return cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
        except ValueError:
            return 0.0

    def score_semantic_similarity(self, answer, golden_answer):
        if not USE_EMBEDDINGS:
            return 0.0
        e1 = np.array(embedder.encode(golden_answer)).reshape(1, -1)
        e2 = np.array(embedder.encode(answer)).reshape(1, -1)
        return float(max(0.0, cosine_similarity(e1, e2)[0][0]))

    def compute_overall_score(self, answer, golden_answer, expected_keywords=None):
        """Returns a score from 0–100."""
        if not answer.strip():
            return 0.0
        if expected_keywords is None:
            expected_keywords = []

        kw     = self.score_keyword_match(answer, expected_keywords)
        length = self.score_length(answer, ideal_length=20)
        tfidf  = self.score_tfidf_similarity(answer, golden_answer)

        if USE_EMBEDDINGS:
            semantic = self.score_semantic_similarity(answer, golden_answer)
            total = kw * 0.2 + length * 0.1 + tfidf * 0.3 + semantic * 0.4
        else:
            semantic = 0.0
            total = kw * 0.2 + length * 0.1 + tfidf * 0.7  # redistribute semantic weight

        return round(total * 100, 2)

    def get_feature_scores(self, answer, golden_answer, expected_keywords=None):
        """Returns individual feature values used for ML prediction."""
        if expected_keywords is None:
            expected_keywords = []
        cos_sim      = self.score_semantic_similarity(answer, golden_answer) if USE_EMBEDDINGS \
                       else self.score_tfidf_similarity(answer, golden_answer)
        golden_words = max(1, len(golden_answer.split()))
        length_ratio = len(answer.split()) / golden_words
        aligned      = self.score_keyword_match(answer, expected_keywords)
        word_count   = len(answer.split())
        return cos_sim, length_ratio, aligned, word_count


print("NLPScorer defined successfully.")

In [ ]:
# =========================
# MODULE 4: Model Training & Comparison
# =========================
NUMERIC_FEATURES = ['cos_similarity', 'length_ratio', 'aligned_score', 'word_count']
TEXT_FEATURE     = 'answer'

X = df[['answer'] + NUMERIC_FEATURES]
y = df['label']   # string labels: 'Good', 'Average', 'Poor'

# Two preprocessors — NB needs non-negative values
preprocessor_standard = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(stop_words='english'), TEXT_FEATURE),
    ("num",   StandardScaler(),                      NUMERIC_FEATURES)
])
preprocessor_minmax = ColumnTransformer(transformers=[
    ("tfidf", TfidfVectorizer(stop_words='english'), TEXT_FEATURE),
    ("num",   MinMaxScaler(),                        NUMERIC_FEATURES)
])

pipelines = {
    "Logistic Regression": Pipeline([("pre", preprocessor_standard),
                                      ("clf", LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))]),
    "SVM":                 Pipeline([("pre", preprocessor_standard),
                                      ("clf", SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42))]),
    "Random Forest":       Pipeline([("pre", preprocessor_standard),
                                      ("clf", RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))]),
    "Complement NB":       Pipeline([("pre", preprocessor_minmax),
                                      ("clf", ComplementNB())])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']

print("\n===== CROSS VALIDATION =====")
final_results = {}

for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    final_results[name] = {
        "Accuracy" : scores['test_accuracy'].mean(),
        "Precision": scores['test_precision_weighted'].mean(),
        "Recall"   : scores['test_recall_weighted'].mean(),
        "F1 Score" : scores['test_f1_weighted'].mean()
    }
    print(f"{name:22s} | Acc: {final_results[name]['Accuracy']:.4f} | F1: {final_results[name]['F1 Score']:.4f}")

comparison_df = pd.DataFrame(final_results).T.round(4)
print("\n===== FINAL COMPARISON TABLE =====")
print(comparison_df)

# Auto-select best model by F1
best_model_name = comparison_df['F1 Score'].idxmax()
print(f"\n*** Best model: {best_model_name} (F1 = {comparison_df.loc[best_model_name, 'F1 Score']:.4f}) ***")

In [ ]:
# =========================
# MODULE 5: Confusion Matrices
# =========================
fig, axes = plt.subplots(1, len(pipelines), figsize=(5 * len(pipelines), 4))

for ax, (name, pipe) in zip(axes, pipelines.items()):
    y_pred = cross_val_predict(pipe, X, y, cv=cv)
    cm = confusion_matrix(y, y_pred, labels=['Good', 'Average', 'Poor'])
    ConfusionMatrixDisplay(cm, display_labels=['Good', 'Average', 'Poor']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)

plt.suptitle("Confusion Matrices — Stratified 5-Fold CV", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# =========================
# MODULE 6: Bar Chart (HTML — renders in Colab/Jupyter)
# =========================
import json
from IPython.display import HTML

chart_data = {
    "models": comparison_df.index.tolist(),
    "metrics": {
        "Accuracy":  comparison_df["Accuracy"].tolist(),
        "Precision": comparison_df["Precision"].tolist(),
        "Recall":    comparison_df["Recall"].tolist(),
        "F1 Score":  comparison_df["F1 Score"].tolist()
    },
    "best": best_model_name
}

html = f"""
<!DOCTYPE html><html><head>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
</head><body style="font-family:sans-serif;padding:20px;">

<div style="display:flex;flex-wrap:wrap;gap:12px;margin-bottom:1.5rem;">
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:120px;">
    <div style="font-size:13px;color:#888;margin-bottom:4px;">Best Model</div>
    <div style="font-size:18px;font-weight:500;" id="best-model">—</div>
  </div>
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:120px;">
    <div style="font-size:13px;color:#888;margin-bottom:4px;">Top Accuracy</div>
    <div style="font-size:18px;font-weight:500;" id="top-acc">—</div>
  </div>
  <div style="background:#f5f5f5;border-radius:8px;padding:1rem;flex:1;min-width:120px;">
    <div style="font-size:13px;color:#888;margin-bottom:4px;">Top F1</div>
    <div style="font-size:18px;font-weight:500;" id="top-f1">—</div>
  </div>
</div>

<div style="display:flex;flex-wrap:wrap;gap:16px;margin-bottom:12px;font-size:12px;color:#666;">
  <span style="display:flex;align-items:center;gap:5px;"><span style="width:10px;height:10px;border-radius:2px;background:#3266ad;display:inline-block;"></span>Accuracy</span>
  <span style="display:flex;align-items:center;gap:5px;"><span style="width:10px;height:10px;border-radius:2px;background:#73726c;display:inline-block;"></span>Precision</span>
  <span style="display:flex;align-items:center;gap:5px;"><span style="width:10px;height:10px;border-radius:2px;background:#1d9e75;display:inline-block;"></span>Recall</span>
  <span style="display:flex;align-items:center;gap:5px;"><span style="width:10px;height:10px;border-radius:2px;background:#d85a30;display:inline-block;"></span>F1 Score</span>
</div>

<div style="position:relative;width:100%;height:380px;">
  <canvas id="modelChart"></canvas>
</div>

<div style="margin-top:1.5rem;overflow-x:auto;">
  <table style="width:100%;border-collapse:collapse;font-size:13px;" id="metrics-table"></table>
</div>

<script>
const data = {json.dumps(chart_data)};
const colors = ['#3266ad','#73726c','#1d9e75','#d85a30'];
const metricKeys = Object.keys(data.metrics);
const datasets = metricKeys.map((k,i) => ({{
  label: k, data: data.metrics[k], backgroundColor: colors[i],
  borderRadius: 3, borderSkipped: false, barPercentage: 0.85, categoryPercentage: 0.8
}}));
new Chart(document.getElementById('modelChart'), {{
  type: 'bar',
  data: {{ labels: data.models, datasets }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ font: {{ size: 12 }}, color: '#888' }} }},
      y: {{ min: 0.5, max: 1.0, grid: {{ color: 'rgba(128,128,128,0.15)' }},
           ticks: {{ font: {{ size: 11 }}, color: '#888', callback: v => v.toFixed(2) }} }}
    }}
  }}
}});
const accVals = data.metrics['Accuracy'];
const f1Vals  = data.metrics['F1 Score'];
const bestIdx = accVals.indexOf(Math.max(...accVals));
const bestF1  = f1Vals.indexOf(Math.max(...f1Vals));
document.getElementById('best-model').textContent = data.best;
document.getElementById('top-acc').textContent    = accVals[bestIdx].toFixed(3);
document.getElementById('top-f1').textContent     = f1Vals[bestF1].toFixed(3);
const table = document.getElementById('metrics-table');
const hdrCols = ['Model', ...metricKeys];
let html = '<thead><tr>' + hdrCols.map(h =>
  `<th style="text-align:${{h==='Model'?'left':'right'}};padding:8px 12px;border-bottom:1px solid #ddd;color:#888;font-weight:500;">${{h}}</th>`
).join('') + '</tr></thead><tbody>';
data.models.forEach((m, mi) => {{
  const isTop = m === data.best;
  html += `<tr style="background:${{isTop?'#f9f9f9':'transparent'}}">`;
  html += `<td style="padding:8px 12px;border-bottom:0.5px solid #eee;font-weight:${{isTop?500:400}}">${{m}}${{isTop?' ★':''}}</td>`;
  metricKeys.forEach(k => {{
    html += `<td style="text-align:right;padding:8px 12px;border-bottom:0.5px solid #eee;">${{data.metrics[k][mi].toFixed(3)}}</td>`;
  }});
  html += '</tr>';
}});
table.innerHTML = html + '</tbody>';
</script></body></html>
"""

display(HTML(html))

In [ ]:
# =========================
# MODULE 7: Export Best Model
# =========================
best_pipe = pipelines[best_model_name]
best_pipe.fit(X, y)
joblib.dump(best_pipe, 'best_interview_model.pkl')
print(f"Best model '{best_model_name}' trained on full dataset and saved as 'best_interview_model.pkl'")

In [ ]:
# =========================
# MODULE 8: Build Question Bank from Dataset
# =========================
# Group unique questions by skill and difficulty
# Each entry has the question text and its ideal_answer as the golden reference

question_bank = {}   # { skill: { difficulty: [ {question, golden_answer, keywords}, ... ] } }

q_df = df[['skill', 'difficulty', 'question', 'ideal_answer']].drop_duplicates('question')

def extract_keywords(text, top_n=8):
    """Extracts simple keyword list from ideal answer using TF-IDF on the single document."""
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    stopwords = {'this','that','with','from','have','been','they','them','when','will',
                 'also','both','each','more','most','into','than','then','over','used',
                 'uses','using','which','where','while','their','there','these','those',
                 'your','like','such','only','some','data','make','made','allows','means'}
    freq = {}
    for w in words:
        if w not in stopwords:
            freq[w] = freq.get(w, 0) + 1
    return [w for w, _ in sorted(freq.items(), key=lambda x: -x[1])][:top_n]

for _, row in q_df.iterrows():
    skill = row['skill']
    diff  = row['difficulty']
    question_bank.setdefault(skill, {}).setdefault(diff, []).append({
        'question':      row['question'],
        'golden_answer': row['ideal_answer'],
        'keywords':      extract_keywords(row['ideal_answer'])
    })

SKILLS = sorted(question_bank.keys())
print("Question bank built.")
print(f"Skills available: {SKILLS}")
for skill in SKILLS:
    counts = {d: len(qs) for d, qs in question_bank[skill].items()}
    print(f"  {skill:15s}: {counts}")

In [ ]:
# =========================
# MODULE 9: Adaptive Interview Engine
# =========================

def get_next_question(skill, difficulty, asked_questions, question_bank):
    """Returns a fresh question for the given skill and difficulty.
    Falls back to other difficulties for the same skill if exhausted."""
    pool = question_bank.get(skill, {}).get(difficulty, [])
    fresh = [q for q in pool if q['question'] not in asked_questions]

    if not fresh:
        # Try other difficulties within the same skill
        all_skill_qs = [q for d, qs in question_bank.get(skill, {}).items()
                        for q in qs if q['question'] not in asked_questions]
        if all_skill_qs:
            return random.choice(all_skill_qs)
        # Last resort: allow repeat
        return random.choice(pool) if pool else None

    return random.choice(fresh)


def adapt_difficulty(current_difficulty, score):
    levels = ["Easy", "Medium", "Hard"]
    idx = levels.index(current_difficulty)
    if score >= 70 and idx < 2:
        print("  [Adaptation] Great answer! ↑ Increasing difficulty.")
        return levels[idx + 1]
    elif score <= 40 and idx > 0:
        print("  [Adaptation] Needs improvement. ↓ Decreasing difficulty.")
        return levels[idx - 1]
    else:
        print("  [Adaptation] Solid answer. → Maintaining current difficulty.")
        return current_difficulty


def predict_label(model, answer, cos_sim, length_ratio, aligned_score, word_count):
    """Runs the trained ML model to classify the answer."""
    try:
        row = pd.DataFrame([{
            'answer':        answer,
            'cos_similarity': cos_sim,
            'length_ratio':  length_ratio,
            'aligned_score': aligned_score,
            'word_count':    word_count
        }])
        return model.predict(row)[0]
    except Exception as e:
        print(f"  ML Prediction Error: {e}")
        return "Unknown"


def label_to_score_adjustment(ml_label):
    return {"Good": +10, "Average": 0, "Poor": -10}.get(ml_label, 0)


def run_interview(question_bank, ml_model, scorer, total_rounds=5):
    print("\n" + "="*55)
    print("  AI-Based Adaptive Mock Interview System")
    print("="*55)

    # Skill selection
    skills = sorted(question_bank.keys())
    print("\nAvailable skills:")
    for i, s in enumerate(skills, 1):
        print(f"  {i}. {s}")

    while True:
        try:
            choice = int(input("\nSelect skill number: "))
            if 1 <= choice <= len(skills):
                selected_skill = skills[choice - 1]
                break
            print(f"  Please enter a number between 1 and {len(skills)}.")
        except ValueError:
            print("  Invalid input. Enter a number.")

    print(f"\nSelected: {selected_skill}")
    print(f"Starting at Medium difficulty | {total_rounds} rounds\n")
    print("-"*55)

    current_difficulty = "Medium"
    asked_questions    = set()
    session_log        = []   # for summary

    for i in range(1, total_rounds + 1):
        print(f"\nRound {i}/{total_rounds} | Skill: {selected_skill} | Level: [{current_difficulty}]")

        q_data = get_next_question(selected_skill, current_difficulty, asked_questions, question_bank)
        if q_data is None:
            print("No more questions available. Ending interview.")
            break

        asked_questions.add(q_data['question'])
        print(f"\nQ: {q_data['question']}")
        print("Your answer (or 'skip' / 'quit'):")

        try:
            user_ans = input("> ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nExiting...")
            break

        if user_ans.lower() in ['quit', 'exit', 'q']:
            print("Interview ended early.")
            break

        if user_ans.lower() == 'skip':
            print("  Skipped.")
            session_log.append({'round': i, 'difficulty': current_difficulty,
                                'question': q_data['question'], 'nlp_score': 0,
                                'ml_label': 'Skipped', 'final_score': 0})
            continue

        print("\n  Evaluating...")

        # NLP scoring
        nlp_score = scorer.compute_overall_score(
            answer         = user_ans,
            golden_answer  = q_data['golden_answer'],
            expected_keywords = q_data['keywords']
        )

        # Feature extraction for ML
        cos_sim, length_ratio, aligned, word_count = scorer.get_feature_scores(
            user_ans, q_data['golden_answer'], q_data['keywords']
        )

        # ML classification
        ml_label = predict_label(ml_model, user_ans, cos_sim, length_ratio, aligned, word_count)

        # Combined final score
        final_score = round(min(100, max(0, nlp_score + label_to_score_adjustment(ml_label))), 2)

        print(f"\n  === Evaluation ===")
        print(f"  NLP Score      : {nlp_score}/100")
        print(f"  ML Label       : {ml_label}")
        print(f"  Final Score    : {final_score}/100")
        print(f"  Ideal Answer   : {q_data['golden_answer'][:180]}...")

        session_log.append({'round': i, 'difficulty': current_difficulty,
                            'question': q_data['question'], 'nlp_score': nlp_score,
                            'ml_label': ml_label, 'final_score': final_score})

        if i < total_rounds:
            current_difficulty = adapt_difficulty(current_difficulty, final_score)

    # Session summary
    print("\n" + "="*55)
    print("  SESSION SUMMARY")
    print("="*55)
    summary_df = pd.DataFrame(session_log)
    if not summary_df.empty:
        print(summary_df[['round','difficulty','ml_label','final_score']].to_string(index=False))
        answered = summary_df[summary_df['ml_label'] != 'Skipped']
        if not answered.empty:
            avg = answered['final_score'].mean()
            print(f"\n  Average Score  : {avg:.1f}/100")
            print(f"  Good answers   : {(answered['ml_label']=='Good').sum()}")
            print(f"  Average answers: {(answered['ml_label']=='Average').sum()}")
            print(f"  Poor answers   : {(answered['ml_label']=='Poor').sum()}")
    print("\nThank you for participating!")
    return summary_df


print("Interview engine ready. Run the next cell to start.")

In [ ]:
# =========================
# MODULE 10: Start Interview
# =========================
scorer   = NLPScorer()
ml_model = joblib.load('best_interview_model.pkl')

session_results = run_interview(
    question_bank = question_bank,
    ml_model      = ml_model,
    scorer        = scorer,
    total_rounds  = 5       # change this to run more/fewer rounds
)